# A1 — Logistična regresija: Telekom

Nadaljujemo z že pripravljenimi podatki. Ena vrstica je naročnik, cilj `odhod` je 1 za odhod in 0 za neodhod.
Časovni horizont oznake v izvornem izvozu ni potrjen, zato napovedi ne opisujemo kot »odhod v naslednjih 30 dneh«.

**Razlaga priprave:** [priprava Telekoma](../01_priprava_podatkov/A1_Priprava_Telekom.ipynb)
in [delitev ter dopolnjevanje manjkajočih vrednosti](../02_preverjanje_modela/A1_Delitev_in_manjkajoce_Telekom.ipynb).
Zaženite celice od začetka do konca. Skupna funkcija po potrebi pripravi podatke. Testne tabele ne beremo.

Podatke po potrebi pripravi `poskrbi_za_modeliranje`; `preberi_tabelo` jih bere s tipi iz sheme.

In [ ]:
#@title Nastavitev okolja (kliknite za prikaz kode)
import subprocess
import sys
from pathlib import Path

REPO_URL = "https://github.com/valira-ai/ui-za-poslovne-aplikacije.git"

if "google.colab" in sys.modules:
    KOREN = Path("/content/ui-za-poslovne-aplikacije")
    if not (KOREN / "utils/orodja.py").is_file():
        if KOREN.exists():
            raise FileNotFoundError(f"Mapa {KOREN} že obstaja, vendar v njej ni utils/orodja.py.")
        subprocess.run(["git", "clone", "-q", REPO_URL, str(KOREN)], check=True)
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "-r", str(KOREN / "requirements.txt")],
        check=True,
    )
else:
    KOREN = Path.cwd().resolve()
    while not (KOREN / "utils/orodja.py").is_file() and KOREN != KOREN.parent:
        KOREN = KOREN.parent
    if not (KOREN / "utils/orodja.py").is_file():
        raise FileNotFoundError("Zaženite zvezek v prenesenem repozitoriju ali njegovi podmapi (manjka utils/orodja.py).")

if str(KOREN) not in sys.path:
    sys.path.insert(0, str(KOREN))
print("Gradivo je v:", KOREN)


In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from scipy.special import expit
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.dummy import DummyClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    log_loss, ConfusionMatrixDisplay,
)

from utils.modeliranje import poskrbi_za_modeliranje, preberi_shemo, preberi_tabelo
PODATKI = poskrbi_za_modeliranje("telekom")
shema = preberi_shemo(PODATKI / "shema.json")
pd.set_option("display.max_rows", 15)
pd.set_option("display.max_columns", 12)
plt.rcParams.update({"figure.figsize": (9, 4), "font.size": 11})

## 1. Naložimo pripravljeni tabeli
Delitev ostane enaka kot pri drugih modelih. Manjkajoče številke so že dopolnjene z **učnimi medianami**,
kategorije pa s stalno oznako `NEZNANO`. Indikatorji povedo, kje je vrednost prvotno manjkala.

In [ ]:
CILJ, KLJUC = shema["cilj"], shema["kljuc"]
train = preberi_tabelo(PODATKI / "train.csv", shema)
val = preberi_tabelo(PODATKI / "val.csv", shema)
assert train[KLJUC].is_unique and val[KLJUC].is_unique
assert set(train[KLJUC]).isdisjoint(val[KLJUC])
assert set(train[CILJ].unique()) == {0, 1}
assert set(val[CILJ].unique()) == {0, 1}
display(pd.DataFrame([
    {"zbirka": ime, "vrstic": len(df), "delež odhodov": df[CILJ].mean()}
    for ime, df in [("učna", train), ("validacijska", val)]
]).round(4))

### Odločitev o vhodih
Za prvi prikaz uporabimo **izvorne značilke in njihove indikatorje manjkajočih vrednosti** iz sheme.
Razmerij, logaritemskih pretvorb in skupin trajanja še ne dodamo; s tem je razlaga bolj pregledna.
Nabor določimo pred pregledom rezultatov na validaciji.

- Številske meritve standardiziramo z učnim povprečjem in standardnim odklonom.
- Indikatorje manjkanja pustimo v obliki 0/1.
- Kategorije kodiramo z one-hot kodiranjem; v spodnjem Pipeline izpustimo prvo kategorijo (`drop="first"`).
- `dohodek_razred` obravnavamo kot kategorijo: številčne oznake ne zagotavljajo enakih razmikov v dohodku.
- ID in cilj nista vhoda v model.

In [ ]:
izvorne = shema["nabori_znacilk"]["izvorni"]
IND = [s for s in shema["skupine_znacilk"]["manjkanje"]
       if s.removesuffix("_manjka") in izvorne]
KAT = [s for s in shema["kategoricne_znacilke"] if s in izvorne] + ["dohodek_razred"]
NUM = [s for s in izvorne if s not in KAT]
ZNACILKE = NUM + IND + KAT
assert len(ZNACILKE) == len(set(ZNACILKE))
assert not set(shema["metapodatki"] + [CILJ]) & set(ZNACILKE)
X_train, y_train = train[ZNACILKE].copy(), train[CILJ].copy()
X_val, y_val = val[ZNACILKE].copy(), val[CILJ].copy()
for X in [X_train, X_val]:
    X[KAT] = X[KAT].astype(str)
    assert not X.isna().any().any()
    assert np.isfinite(X[NUM + IND].to_numpy(dtype=float)).all()
    assert X[IND].isin([0, 1]).all().all()
display(pd.DataFrame({
    "obravnava": ["StandardScaler", "indikatorji 0/1", "OneHotEncoder"],
    "število vhodnih značilk": [len(NUM), len(IND), len(KAT)],
}))

## 2. Pripravimo Pipeline in naučimo model
`ColumnTransformer` pripravi različne vrste stolpcev, `Pipeline` pa pripravo poveže z modelom.
`fit` nauči skaliranje, slovar kategorij in koeficiente **samo na učnih vrsticah**.
`predict_proba` na validaciji uporabi že naučene korake.

Uporabimo privzeto regularizacijo L2 in `C=1`. Kasneje spremenimo samo `C`.
Pri neznani kategoriji `handle_unknown="ignore"` ustvari same ničle za njeno skupino,
enako kot pri izpuščeni referenčni kategoriji; to ni naučena kategorija `NEZNANO`.

Imputacija je bila izvedena v prejšnjem poglavju. Ta Pipeline zato pričakuje že dopolnjene tabele.
Za prečno preverjanje bi obnovili nedopolnjene učne vrstice in imputacijo vključili v vsak učni krog.

In [ ]:
def sestavi_model(znacilke, C=1.0):
    # Vrste vhodov določimo iz sheme tudi za razširjeni nabor v zadnji vaji.
    kat = [s for s in znacilke if s in shema["kategoricne_znacilke"] or s == "dohodek_razred"]
    ind = [s for s in znacilke if s in shema["skupine_znacilk"]["manjkanje"]]
    num = [s for s in znacilke if s not in kat + ind]
    # Nova priprava in nov model za vsak neodvisen poskus.
    priprava = ColumnTransformer([
        ("num", StandardScaler(), num),
        ("ind", "passthrough", ind),
        ("kat", OneHotEncoder(handle_unknown="ignore", drop="first", sparse_output=False),
         kat),
    ], remainder="drop")
    return Pipeline([
        ("priprava", priprava),
        ("model", LogisticRegression(C=C, solver="lbfgs", max_iter=3000, tol=1e-6)),
    ])

model = sestavi_model(ZNACILKE)
model.fit(X_train, y_train)
assert list(model.classes_) == [0, 1]
assert model.named_steps["model"].n_iter_.max() < 3000, "Preverite konvergenco modela."
priprava = model.named_steps["priprava"]
np.testing.assert_allclose(
    priprava.named_transformers_["num"].mean_, X_train[NUM].mean().to_numpy()
)
print("Število iteracij:", model.named_steps["model"].n_iter_[0])
print("Število stolpcev po pripravi:", len(priprava.get_feature_names_out()))

encoder = priprava.named_transformers_["kat"]
display(pd.DataFrame({
    "značilka": KAT,
    "izpuščena kategorija": [kategorije[i] for kategorije, i in zip(encoder.categories_, encoder.drop_idx_)],
}))


## 3. Preverimo napovedi
Verjetnost se nanaša na razred **1 = odhod**. Prag za prikaz je 0,5.
Izhodiščni model vsakemu naročniku pripiše učni delež odhodov; ne uporablja značilk.
Log loss meri kakovost verjetnosti, ostale metrike v tabeli pa razrede pri izbranem pragu.

In [ ]:
PRAG = 0.5

def izmeri(y, p, prag=None):
    prag = PRAG if prag is None else prag
    napoved = (np.asarray(p) >= prag).astype(int)
    return {
        "accuracy": accuracy_score(y, napoved),
        "precision": precision_score(y, napoved, zero_division=0),
        "recall": recall_score(y, napoved, zero_division=0),
        "F1": f1_score(y, napoved, zero_division=0),
        "log loss": log_loss(y, p, labels=[0, 1]),
    }

p_train = model.predict_proba(X_train)[:, 1]
p_val = model.predict_proba(X_val)[:, 1]
izh_model = DummyClassifier(strategy="prior").fit(X_train, y_train)
p_izh = izh_model.predict_proba(X_val)[:, 1]
rezultati = pd.DataFrame([
    {"model / zbirka": "Izhodišče / validacija", **izmeri(y_val, p_izh)},
    {"model / zbirka": "Logistična / učenje", **izmeri(y_train, p_train)},
    {"model / zbirka": "Logistična / validacija", **izmeri(y_val, p_val)},
]).set_index("model / zbirka")
display(rezultati.round(4))
print("Če ni pozitivnih napovedi, sta precision in F1 po dogovoru v izpisu 0.")

In [ ]:
fig, ax = plt.subplots(figsize=(5.5, 4.5))
ConfusionMatrixDisplay.from_predictions(
    y_val, (p_val >= PRAG).astype(int), labels=[0, 1],
    display_labels=["Brez odhoda", "Odhod"], cmap="Blues", colorbar=False, ax=ax,
)
ax.set(title="Validacija · prag 0,5", xlabel="Napoved", ylabel="Dejanski izid")
plt.tight_layout()
plt.show()
display(pd.DataFrame({
    "dejanski odhod": y_val.iloc[:8].to_numpy(),
    "verjetnost odhoda": p_val[:8],
    "napovedani razred": (p_val[:8] >= PRAG).astype(int),
}).round(3))

### Zakaj samozavestna napaka dobi veliko kazen?
Za en primer izračunamo $\ell=-[y\ln(p)+(1-y)\ln(1-p)]$.
Najprej uporabimo štiri **umetne primere**, nato isti izračun ponovimo na dejanskih validacijskih napovedih.

In [ ]:
umetni = pd.DataFrame({"y": [1, 1, 0, 0], "p": [0.9, 0.1, 0.1, 0.9]})
umetni["pravilno pri 0,5"] = (umetni.p >= PRAG).astype(int).eq(umetni.y)
umetni["izguba"] = -(umetni.y * np.log(umetni.p) + (1 - umetni.y) * np.log1p(-umetni.p))
display(umetni.round(3))

# Za numerično stabilnost se izognemo log(0).
eps = np.finfo(float).eps
p_varno = np.clip(p_val, eps, 1 - eps)
izgube = -(y_val.to_numpy() * np.log(p_varno) + (1 - y_val.to_numpy()) * np.log1p(-p_varno))
np.testing.assert_allclose(izgube.mean(), log_loss(y_val, p_val))
pregled_napak = pd.DataFrame({"y": y_val, "p": p_val, "izguba": izgube})
display(pregled_napak.nlargest(5, "izguba").round(4))
print(f"Povprečna validacijska izguba: {izgube.mean():.4f}")

**Vprašanja:** Je model boljši od izhodišča? Katere napake dela pri pragu 0,5?
Zakaj log loss razlikuje med p=0,51 in p=0,99, čeprav oba dasta isti razred?
Prag ni bil poslovno optimiziran; ocenjene verjetnosti tudi niso samodejno dobro kalibrirane.

## 4. Kako razlagamo koeficiente?
Koeficienti opisujejo **oceno z oziroma logaritem obetov**, ne spremembe verjetnosti v odstotnih točkah.
Tabela mora slediti istemu vrstnemu redu kot pripravljeni stolpci.

In [ ]:
lr = model.named_steps["model"]
imena = priprava.get_feature_names_out()
beta = pd.Series(lr.coef_[0], index=imena, name="beta")
b0 = float(lr.intercept_[0])
koeficienti = beta.to_frame()
koeficienti["absolutna vrednost"] = beta.abs()
koeficienti["vrsta"] = [s.split("__", 1)[0] for s in imena]
print(f"Prestreznik b₀: {b0:.4f}")
print("Celotna tabela je na voljo v spremenljivki koeficienti.")
display(koeficienti.head(8).round(4))

### Numerične značilke: povečanje za en učni standardni odklon
Večji |β| pomeni večjo spremembo logaritma obetov za takšno povečanje, ob nespremenjenih drugih vhodih.
`exp(β)` je faktor, s katerim se pomnožijo obeti. To ni splošna lestvica napovedne pomembnosti.
Standardni odkloni so izračunani na **dopolnjenih učnih podatkih**.

In [ ]:
num_koef = pd.DataFrame({
    "značilka": NUM,
    "učni standardni odklon": priprava.named_transformers_["num"].scale_,
    "beta": beta.loc[["num__" + s for s in NUM]].to_numpy(),
}).set_index("značilka")
num_koef["razmerje obetov za +1 SD"] = np.exp(num_koef["beta"])
vrstni_red = num_koef["beta"].abs().sort_values(ascending=False).index
display(num_koef.loc[vrstni_red].head(12).round(4))
graf = num_koef.loc[vrstni_red[:12]].sort_values("beta")
fig, ax = plt.subplots(figsize=(10, 5))
ax.barh(graf.index, graf.beta, color=np.where(graf.beta > 0, "#c45151", "#168a87"))
ax.axvline(0, color="grey", linewidth=1)
ax.set(title="Koeficienti numeričnih značilk · 12 največjih |β|", xlabel="Sprememba logaritma obetov za +1 SD")
plt.tight_layout()
plt.show()

In [ ]:
# Dva konkretna primera; po želji zamenjajte značilki.
for znacilka in ["mesecev_narocnine", "povp_mesecni_prihodek"]:
    r = num_koef.loc[znacilka]
    print(f"{znacilka}: +{r['učni standardni odklon']:.2f} izvirnih enot "
          f"→ sprememba z {r['beta']:+.3f} "
          f"→ obeti × {r['razmerje obetov za +1 SD']:.3f}")

# Indikatorji niso standardizirani: primerjamo 0 in 1.
ime_ind = "stevilo_avtomobilov_manjka"
b_ind = beta["ind__" + ime_ind]
print(f"{ime_ind}: prehod 0 → 1, obeti × {np.exp(b_ind):.3f}.")

### Kategorije: primerjamo dve kategoriji iste značilke
Za izpuščeno kategorijo vzamemo **β = 0**; za prehod **A → B** uporabimo razliko **βB − βA**.
Razmerje obetov je **exp(βB − βA)**. Za prikaz lahko izberemo tudi drugo referenco; modela ne učimo ponovno.
Indikatorjev one-hot ne standardiziramo.

Prikažemo `dualband`. Oznak iz vira ne prevajamo v poslovne pomene, ki niso potrjeni.
Kategorija `NEZNANO` pomeni manjkajoč podatek, ne lastnosti naprave.

In [ ]:
encoder = priprava.named_transformers_["kat"]
kat_imena = encoder.get_feature_names_out(KAT)
kat_beta = pd.Series(beta.loc[["kat__" + s for s in kat_imena]].to_numpy(), index=kat_imena)
kat_vrstice = []
zacetek = 0
for znacilka, kategorije, izpuscena in zip(KAT, encoder.categories_, encoder.drop_idx_):
    for i, kategorija in enumerate(kategorije):
        # Izpuščena kategorija nima svojega stolpca; njen prispevek je 0.
        b = 0.0 if i == izpuscena else kat_beta.iloc[zacetek]
        kat_vrstice.append({"značilka": znacilka, "kategorija": kategorija, "beta": b})
        if i != izpuscena:
            zacetek += 1
assert zacetek == len(kat_beta)
kat_tabela = pd.DataFrame(kat_vrstice)

KATEGORICNA = "dualband"
REFERENCA = "Y"
primerjava_kat = kat_tabela.loc[kat_tabela["značilka"].eq(KATEGORICNA)].copy()
b_ref = primerjava_kat.loc[primerjava_kat.kategorija.eq(REFERENCA), "beta"].item()
primerjava_kat["razlika glede na referenco"] = primerjava_kat.beta - b_ref
primerjava_kat["razmerje obetov"] = np.exp(primerjava_kat["razlika glede na referenco"])
primerjava_kat["učnih primerov"] = primerjava_kat.kategorija.map(X_train[KATEGORICNA].value_counts())
print(f"Primerjava: {KATEGORICNA}, referenca = {REFERENCA}")
display(primerjava_kat.round(4))

# Preverimo razliko tudi prek napovedi modela za dve sicer enaki vrstici.
par = pd.concat([X_val.iloc[[0]], X_val.iloc[[0]]], ignore_index=True)
par[KATEGORICNA] = ["Y", "N"]
razlika_z = np.diff(model.decision_function(par)).item()
razlika_beta = primerjava_kat.loc[primerjava_kat.kategorija.eq("N"), "razlika glede na referenco"].item()
np.testing.assert_allclose(razlika_z, razlika_beta, atol=1e-10)
print(f"Preverjeno Y → N: sprememba z = {razlika_z:.4f}, obeti × {np.exp(razlika_z):.4f}")

### Povezane značilke otežijo razlago
Koeficient opisuje spremembo ob nespremenjenih drugih vhodih. Če sta dve meritvi zelo povezani,
takšna sprememba ni nujno pogosta ali smiselna v resničnih podatkih. Model lahko vpliv razdeli med obe.
Tudi velik koeficient ne zagotavlja boljše napovedi na novih podatkih.

Spodaj pogledamo tri najmočneje povezane pare **samo na učnih podatkih**.
Koeficienti niso dokaz vzročnosti. Pri redkih kategorijah je razlaga dodatno negotova.

In [ ]:
korelacije = X_train[NUM].corr()
maska = np.triu(np.ones(korelacije.shape, dtype=bool), k=1)
pari = korelacije.where(maska).stack().rename("korelacija").reset_index()
pari.columns = ["značilka A", "značilka B", "korelacija"]
pari["absolutno"] = pari.korelacija.abs()
display(pari.nlargest(3, "absolutno").drop(columns="absolutno").round(3))

## 5. Razložimo eno napoved 
Izberemo prvo validacijsko vrstico, **brez izbire po znanem izidu**. Izračunamo pripravljene vhode,
prispevke β × x, oceno z in sigmoid. Nato rezultat primerjamo z `predict_proba`.
Prestreznik je ocena pri vseh pripravljenih vhodih nič: numerične značilke so na učnih povprečjih,
indikatorji so 0, kategorije pa referenčne. To ni nujno dejanski naročnik ali povprečna napoved populacije.

In [ ]:
VRSTICA = 0  # Za prikaz drugega naročnika spremenite indeks in ponovite ta del.
primer = X_val.iloc[[VRSTICA]]
x_pripravljen = priprava.transform(primer)[0]
prispevki = pd.DataFrame({"pripravljena vrednost": x_pripravljen, "beta": beta.to_numpy()}, index=imena)
prispevki["prispevek k z"] = prispevki["pripravljena vrednost"] * prispevki.beta
z_rocno = b0 + prispevki["prispevek k z"].sum()
p_rocno = float(expit(z_rocno))
p_model = model.predict_proba(primer)[0, 1]
np.testing.assert_allclose(z_rocno, model.decision_function(primer)[0], atol=1e-10)
np.testing.assert_allclose(p_rocno, p_model, atol=1e-12)
display(primer[["mesecev_narocnine", "povp_mesecni_prihodek", "dualband", "stevilo_avtomobilov"]])
print(f"b₀ = {b0:.4f}; vsota prispevkov = {prispevki['prispevek k z'].sum():.4f}")
print(f"z = {z_rocno:.4f}; sigmoid(z) = {p_rocno:.4f}; predict_proba = {p_model:.4f}")
print(f"Napoved pri pragu 0,5: {int(p_rocno >= PRAG)}; dejanski izid: {y_val.iloc[VRSTICA]}")

In [ ]:
najvecji = prispevki["prispevek k z"].abs().nlargest(8).index
prikaz = prispevki.loc[najvecji, "prispevek k z"].sort_values()
# Preostalih prispevkov ne izpustimo iz seštevka.
ostali = prispevki.drop(index=najvecji)["prispevek k z"].sum()
prikaz = pd.concat([pd.Series({"Prestreznik b₀": b0}), prikaz, pd.Series({"Ostali prispevki skupaj": ostali})])
np.testing.assert_allclose(prikaz.sum(), z_rocno, atol=1e-10)
fig, ax = plt.subplots(figsize=(11, 5.5))
ax.barh(prikaz.index, prikaz, color=np.where(prikaz > 0, "#c45151", "#168a87"))
ax.axvline(0, color="grey", linewidth=1)
ax.set(title=f"Razlaga napovedi · z = {z_rocno:.3f}, p = {p_rocno:.3f}", xlabel="Prispevki k logaritemski oceni z")
ax.invert_yaxis()
plt.tight_layout()
plt.show()

**Kaj vidimo?** Pozitivni prispevki zvišujejo z, negativni ga znižujejo.
Seštevajo se prispevki v logitu, **ne spremembe verjetnosti**. To je natančna razčlenitev izračuna tega modela. Delitev med prestreznik in kategorične prispevke je odvisna od kodiranja.

## 6. Vpliv regularizacije
Primerjamo **C = 0,001, 1 in 100**. Manjši C pomeni močnejšo kazen L2.
Podatki, priprava in ostale nastavitve ostanejo enaki. `C=1` smo že naučili in ga ponovno uporabimo.

Pogledamo normo koeficientov in učni ter validacijski log loss. Močnejša kazen omeji skupno velikost koeficientov,
ne zagotavlja pa, da se bo vsak posamezen koeficient zmanjšal ali da bo validacija boljša.

In [ ]:
C_VREDNOSTI = [0.001, 1.0, 100.0]
modeli_C = {1.0: model}
vrstice_C, beta_C = [], {}
for C in C_VREDNOSTI:
    if C not in modeli_C:
        modeli_C[C] = sestavi_model(ZNACILKE, C=C).fit(X_train, y_train)
    kandidat = modeli_C[C]
    klasifikator = kandidat.named_steps["model"]
    assert klasifikator.n_iter_.max() < klasifikator.max_iter
    beta_C[C] = klasifikator.coef_[0]
    vrstice_C.append({
        "C": C,
        "norma koeficientov L2": np.linalg.norm(klasifikator.coef_[0]),
        "učni log loss": log_loss(y_train, kandidat.predict_proba(X_train)[:, 1]),
        "validacijski log loss": log_loss(y_val, kandidat.predict_proba(X_val)[:, 1]),
    })
regularizacija = pd.DataFrame(vrstice_C).set_index("C")
display(regularizacija.round(5))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
regularizacija[["učni log loss", "validacijski log loss"]].plot(ax=axes[0], marker="o", logx=True)
axes[0].set(xlabel="C · manjši pomeni močnejšo kazen", ylabel="Log loss", title="Kakovost verjetnosti")
regularizacija["norma koeficientov L2"].plot(ax=axes[1], marker="o", logx=True, color="#168a87")
axes[1].set(xlabel="C", ylabel="Norma L2", title="Skupna velikost koeficientov")
plt.tight_layout()
plt.show()
# Iste značilke v vseh treh modelih; izbor za prikaz temelji na modelu C=1.
izbrani = ["num__" + s for s in vrstni_red[:5]]
display(pd.DataFrame(beta_C, index=imena).loc[izbrani].rename_axis(columns="C").round(4))

**Vprašanja:** Ali močnejša regularizacija izboljša validacijski log loss? Ali se spremeni tudi znak katerega koeficienta?
Če se koeficienti precej spremenijo, kakovost napovedi pa malo, kaj to pomeni za razlago?
Razlike na eni validacijski delitvi niso dokaz stabilne prednosti.

Model `model` ostaja izhodiščni C=1. V naslednji vaji ohranimo ta C, da spreminjamo samo nabor značilk.

## 7. Vaja: ali število avtomobilov pomaga?
Vrednost pri številnih naročnikih manjka. Po dopolnjevanju to vidimo v indikatorju `stevilo_avtomobilov_manjka`.

**Pred izvajanjem zapišite pričakovanje:** katera različica bo boljša in zakaj?

1. Število avtomobilov **in** indikator manjkanja.
2. **Samo indikator**, brez števila avtomobilov.
3. **Brez obeh** stolpcev.

V vseh treh različicah ostanejo iste vrstice, isti C=1 in ista pravila priprave. Za spremenjena nabora model
ponovno naučimo; stolpca pri napovedovanju ne nastavimo samo na nič. Prva različica je že naučeni model.

In [ ]:
AVTO, MANJKA = "stevilo_avtomobilov", "stevilo_avtomobilov_manjka"
display(pd.DataFrame([
    {"zbirka": ime, "delež prvotno manjkajočih": X[MANJKA].mean()}
    for ime, X in [("učna", X_train), ("validacijska", X_val)]
]).round(4))
nabori = {
    "Vrednost in indikator": ZNACILKE,
    "Samo indikator": [s for s in ZNACILKE if s != AVTO],
    "Brez obeh": [s for s in ZNACILKE if s not in [AVTO, MANJKA]],
}
modeli_avto = {"Vrednost in indikator": model}
rezultati_avto = []
for ime, nabor in nabori.items():
    if ime not in modeli_avto:
        modeli_avto[ime] = sestavi_model(nabor, C=1.0).fit(X_train[nabor], y_train)
    kandidat = modeli_avto[ime]
    assert kandidat.named_steps["model"].n_iter_.max() < 3000
    p = kandidat.predict_proba(X_val[nabor])[:, 1]
    rezultati_avto.append({"različica": ime, "značilk": len(nabor), **izmeri(y_val, p)})
primerjava_avto = pd.DataFrame(rezultati_avto).set_index("različica")
primerjava_avto["sprememba log loss glede na obe"] = (
    primerjava_avto["log loss"] - primerjava_avto.loc["Vrednost in indikator", "log loss"]
)
display(primerjava_avto.round(5))

### Razprava in predlog razlage rezultatov
- **Vrednost in indikator → samo indikator:** kaj prispeva podatek o številu avtomobilov poleg ostalih vhodov?
- **Samo indikator → brez obeh:** ali informacija o manjkanju pomaga tudi brez same vrednosti?
- Pri **log loss je manj bolje**. Pozitivna sprememba v zadnjem stolpcu pomeni slabši rezultat od prve različice.
- Precision in recall se lahko premakneta v različni smeri; primerjamo ju pri istem pragu.
- Majhna razlika na eni delitvi ni dokaz, da bo različica vedno boljša. Koristnost je odvisna tudi od ostalih značilk.

**Oddaja vaje:** izbrana različica, številčna primerjava z izhodiščem in ena omejitev sklepa.
Test ostane zaprt do zaključka izbire modela in nastavitev. Za zanesljivejšo primerjavo pozneje uporabimo prečno preverjanje
z učenjem imputacije, kodiranja in skaliranja znotraj vsakega kroga.

## 8. Vaja: kaj izgubimo brez podatka o starosti naprave?
Pri številu avtomobilov so bile razlike majhne. Zdaj preizkusimo `dni_z_napravo`:
koliko dni ima naročnik trenutno napravo.

**Pred izvajanjem zapišite pričakovanje:** ali se bo model brez tega podatka poslabšal bolj
kot brez števila avtomobilov? Kaj pričakujete glede na predhodno razlago koeficientov?

Primerjamo isti model z izbrano značilko in brez nje. Odstranimo **samo vrednost**;
indikator manjkanja in vse ostale značilke ostanejo. Model brez značilke ponovno naučimo
na istih učnih vrsticah, z istim `C=1` in istimi pravili priprave. Testa ne odpremo.

Glavna mera je **validacijski log loss: manj je bolje**. Sprememba glede na model z značilko
pokaže, koliko smo z odstranitvijo izgubili ali pridobili.

Pogledamo še **točnost, natančnost, priklic in F1** pri istem pragu kot prej (`PRAG = 0.5`).
Te mere prikažemo v odstotkih, spremembe pa v **odstotnih točkah**; pri njih je več bolje.
Log loss meri kakovost verjetnosti, ostale mere pa razrede po uporabi praga.


In [ ]:
IZBRANA_ZNACILKA = "dni_z_napravo"  # Nato poskusite še "povp_minute" ali "mesecev_narocnine".
assert IZBRANA_ZNACILKA in NUM, "Izberite numerično značilko iz začetnega nabora."

nabor_brez = [s for s in ZNACILKE if s != IZBRANA_ZNACILKA]
model_brez_znacilke = sestavi_model(nabor_brez, C=1.0).fit(X_train[nabor_brez], y_train)
assert model_brez_znacilke.named_steps["model"].n_iter_.max() < 3000

p_z_znacilko = model.predict_proba(X_val[ZNACILKE])[:, 1]
p_brez_znacilke = model_brez_znacilke.predict_proba(X_val[nabor_brez])[:, 1]
primerjava_znacilke = pd.DataFrame({
    "Z značilko": izmeri(y_val, p_z_znacilko),
    "Brez značilke": izmeri(y_val, p_brez_znacilke),
})
loss_z, loss_brez = primerjava_znacilke.loc["log loss", ["Z značilko", "Brez značilke"]]
primerjava_loss = primerjava_znacilke.loc[["log loss"]].copy()
primerjava_loss["Sprememba"] = primerjava_loss["Brez značilke"] - primerjava_loss["Z značilko"]
print(f"Izbrana značilka: {IZBRANA_ZNACILKA}; log loss: manj je bolje.")
display(primerjava_loss.round(5))

IMENA_MER = {"accuracy": "Točnost", "precision": "Natančnost", "recall": "Priklic", "F1": "F1"}
primerjava_mer = primerjava_znacilke.loc[list(IMENA_MER)].rename(index=IMENA_MER) * 100
primerjava_mer.columns = ["Z značilko (%)", "Brez značilke (%)"]
primerjava_mer["Sprememba (o. t.)"] = primerjava_mer["Brez značilke (%)"] - primerjava_mer["Z značilko (%)"]
print(f"Mere pri pragu {PRAG:g}: več je bolje; o. t. = odstotne točke.")
display(primerjava_mer.round(3))


In [ ]:
# V obeh poskusih odstranimo samo vrednost; indikator manjkanja ostane.
sprememba_avto = (primerjava_avto.loc["Samo indikator", "log loss"]
                  - primerjava_avto.loc["Vrednost in indikator", "log loss"])
spremembe = pd.Series(
    [sprememba_avto, loss_brez - loss_z],
    index=["Brez števila avtomobilov (vaja 7)", f"Brez {IZBRANA_ZNACILKA} (vaja 8)"],
)
fig, ax = plt.subplots(figsize=(9, 3))
stolpci = ax.barh(spremembe.index, spremembe.to_numpy(), color="#2993AD")
ax.bar_label(stolpci, labels=[f"{v:+.5f}" for v in spremembe], padding=5)
ax.axvline(0, color="grey", linewidth=1)
ax.invert_yaxis()
ax.margins(x=0.25)
ax.set(title="Kaj se spremeni, ko model ponovno naučimo brez značilke?",
       xlabel="Sprememba validacijskega log loss · pozitivno = slabše")
plt.tight_layout()
plt.show()

# Pri teh merah negativna sprememba pomeni poslabšanje.
razlike_mer = primerjava_mer["Sprememba (o. t.)"]
fig, ax = plt.subplots(figsize=(9, 3))
stolpci = ax.barh(razlike_mer.index, razlike_mer.to_numpy(),
                  color=np.where(razlike_mer < 0, "#c45151", "#168a87"))
ax.bar_label(stolpci, labels=[f"{v:+.3f}" for v in razlike_mer], padding=5)
ax.axvline(0, color="grey", linewidth=1)
ax.invert_yaxis()
ax.margins(x=0.25)
ax.set(title=f"Brez {IZBRANA_ZNACILKA} · prag {PRAG:g}",
       xlabel="Sprememba v odstotnih točkah · negativno = slabše")
plt.tight_layout()
plt.show()


### Razprava
- Katera odstranitev bolj poslabša log loss? Uporabite spremembo iz tabele, ne le videza grafa.
- Ali se vse štiri mere pri pragu 0,5 poslabšajo enako? Katera se spremeni najbolj?
- Ali model brez podatka o starosti naprave še vedno zna napovedovati? Katere druge značilke mu lahko pomagajo?
- Zamenjajte `IZBRANA_ZNACILKA` z eno od predlaganih in ponovno izvedite obe celici. Se vaše pričakovanje potrdi?

<details>
<summary>Primer razlage za dni_z_napravo</summary>

V preverjenem zagonu z `C=1` je log loss narasel približno z **0,67476 na 0,67781**
(sprememba **+0,00304**). Pri odstranitvi števila avtomobilov je bila sprememba približno **+0,00003**.
Pri pragu 0,5 se točnost zmanjša približno z **58,07 % na 57,58 %**, natančnost
z **57,90 % na 57,48 %**, priklic z **56,39 % na 55,39 %** in F1 z **57,14 % na 56,42 %**.
Največ se spremeni priklic: model najde približno **1 odstotno točko manj** strank, ki odidejo.
Podatek o starosti naprave torej v tem poskusu prispeva k napovedi poleg ostalih značilk;
velikost spremembe pa je odvisna od izbrane mere.

To ni dokaz vzročnega vpliva starosti naprave na odhod. Rezultat velja za ta model,
ta nabor vhodov in to validacijsko delitev; povezane značilke lahko del informacije nadomestijo.

</details>

**Oddaja vaje:** izbrana značilka, sprememba validacijskega log loss, sprememba ene od mer pri pragu 0,5 in ena poved o tem, kaj rezultat pove.


## 9. Vaja: prilagodimo nastavitve modela
Zdaj sami izberite nastavitve in primerjajte rezultat z začetnim modelom
(**47 značilk, C=1, prag 0,5**).

- **`C_POSKUS`**: manjša vrednost pomeni močnejšo regularizacijo L2. Poskusite `0.001`, `0.01`, `1.0` in `100.0`.
- **`VSE_ZNACILKE`**: `False` uporabi začetnih 47, `True` vseh 64 pripravljenih značilk,
  vključno z razmerji, pretvorbami in njihovimi indikatorji manjkanja. ID in cilj ostaneta izključena.
- **`PRAG_POSKUS`**: poskusite `0.3`, `0.5` in `0.7`. Prag je nastavitev odločitve po napovedi,
  **ni hiperparameter učenja**. Ob istem modelu ne spremeni verjetnosti ali log loss.

**Pred vsakim poskusom zapišite pričakovanje.** Najprej spreminjajte samo `C`, nato pri `C=1`
preklopite nabor značilk. Nazadnje pri izbranem modelu spreminjajte samo prag.
Po vsaki spremembi ponovno zaženite spodnji dve celici in si zabeležite rezultat.


In [ ]:
# Spremenite eno nastavitev naenkrat.
C_POSKUS = 0.001
VSE_ZNACILKE = False
PRAG_POSKUS = 0.5

assert np.isfinite(C_POSKUS) and C_POSKUS > 0, "C mora biti pozitivno končno število."
assert 0 < PRAG_POSKUS < 1, "Prag naj bo med 0 in 1."

znacilke_poskus = list(shema["znacilke"]) if VSE_ZNACILKE else list(ZNACILKE)
assert len(znacilke_poskus) == len(set(znacilke_poskus))
assert not set(znacilke_poskus) & set(shema["metapodatki"] + [CILJ])
kat_poskus = [s for s in znacilke_poskus
              if s in shema["kategoricne_znacilke"] or s == "dohodek_razred"]
X_train_poskus = train[znacilke_poskus].copy()
X_val_poskus = val[znacilke_poskus].copy()
for X in [X_train_poskus, X_val_poskus]:
    X[kat_poskus] = X[kat_poskus].astype(str)
    assert not X.isna().any().any()

model_poskus = sestavi_model(
    znacilke_poskus, C=C_POSKUS,
).fit(X_train_poskus, y_train)
klasifikator_poskus = model_poskus.named_steps["model"]
assert klasifikator_poskus.n_iter_.max() < klasifikator_poskus.max_iter, "Preverite konvergenco modela."
p_train_poskus = model_poskus.predict_proba(X_train_poskus)[:, 1]
p_val_poskus = model_poskus.predict_proba(X_val_poskus)[:, 1]
print(f"Poskus: {len(znacilke_poskus)} značilk, C={C_POSKUS:g}, "
      f"prag={PRAG_POSKUS:g}")


In [ ]:
# Začetni model in njegove nastavitve ostanejo na voljo za primerjavo.
vrstice_poskusa = []
for ime, zbirka, y, p, prag in [
    ("Začetni model", "učna", y_train, model.predict_proba(X_train)[:, 1], 0.5),
    ("Začetni model", "validacijska", y_val, model.predict_proba(X_val)[:, 1], 0.5),
    ("Poskus", "učna", y_train, p_train_poskus, PRAG_POSKUS),
    ("Poskus", "validacijska", y_val, p_val_poskus, PRAG_POSKUS),
]:
    vrstice_poskusa.append({"model": ime, "zbirka": zbirka, **izmeri(y, p, prag=prag)})
rezultati_poskusa = pd.DataFrame(vrstice_poskusa).set_index(["model", "zbirka"])

imena_mer_poskusa = {"accuracy": "Točnost (%)", "precision": "Natančnost (%)",
                     "recall": "Priklic (%)", "F1": "F1 (%)"}
prikaz_poskusa = rezultati_poskusa.copy()
prikaz_poskusa[list(imena_mer_poskusa)] *= 100
print("Točnost, natančnost, priklic in F1: več je bolje; log loss: manj je bolje.")
display(prikaz_poskusa.rename(columns=imena_mer_poskusa).round(5))

razlika_poskusa = (rezultati_poskusa.loc[("Poskus", "validacijska")]
                   - rezultati_poskusa.loc[("Začetni model", "validacijska")])
razlika_poskusa[list(imena_mer_poskusa)] *= 100
razlika_poskusa = razlika_poskusa.rename(index={
    k: v.replace("(%)", "(o. t.)") for k, v in imena_mer_poskusa.items()
})
print("Sprememba na validaciji glede na začetni model (o. t. = odstotne točke):")
display(razlika_poskusa.to_frame("Poskus − začetni model").round(5))


### Razprava
- Kaj se ob močnejši regularizaciji zgodi z učno izgubo? Ali se validacijska spremeni v isti smeri?
- Ali vseh 64 značilk pomaga tudi pri drugem `C`? Primerjajte nabora pri enakih ostalih nastavitvah.
- Spreminjajte samo prag: zakaj se točnost, natančnost, priklic in F1 spreminjajo, log loss pa ostaja enak?
- Kateri poskus bi izbrali za kakovost verjetnosti in katerega za večji priklic? Utemeljite kompromis.

**Oddaja vaje:** zabeležite vsaj tri kombinacije nastavitev in njihove validacijske metrike.
Za izbrano kombinacijo navedite še učni log loss ter razložite, kaj ste pridobili in kaj izgubili
v primerjavi z začetnim modelom. Boljši učni rezultat sam po sebi ne pomeni boljšega modela.

Nastavitve izbiramo na validaciji. Po številnih poskusih lahko tudi njene posebnosti vplivajo na izbiro;
rezultat zato še ni neodvisna končna ocena. Test ostane zaprt do končne primerjave modelov.


## Kaj smo povezali s prosojnicami?
Značilke in koeficienti določijo z; sigmoid ga pretvori v verjetnost. Učenje zmanjšuje logaritemsko izgubo,
regularizacija pa omejuje koeficiente. Razložili smo numerične spremembe, prehode med kategorijami in izračun ene napovedi.
Napovedno korist stolpca smo preverili s ponovnim učenjem in validacijo.